In [1]:
# Install if needed (uncomment the line below)
# !pip install requests beautifulsoup4 lxml pandas matplotlib

import io
import json
import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
from pathlib import Path

# Folder to store collected data
DATA = Path("data")
DATA.mkdir(exist_ok=True)

print("Setup complete. Data will be saved in:", DATA.resolve())

Setup complete. Data will be saved in: C:\Users\Chirantana D Naik\Videos\Data Science\Lab-1\data


In [2]:
# --- Step 2a: Ask the API for the current weather ---

URL = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 12.9716,        # Bengaluru
    "longitude": 77.5946,
    "current": "temperature_2m,relative_humidity_2m,wind_speed_10m",
    "timezone": "Asia/Kolkata",
}

try:
    response = requests.get(URL, params=params, timeout=20)
    print("Status code:", response.status_code)
    weather = response.json()
except Exception as e:
    weather = None
    print("No internet:", e)

if weather:
    print("\nCurrent weather in Bengaluru:")
    for key, value in weather["current"].items():
        print(f"  {key:22s}: {value}")

Status code: 200

Current weather in Bengaluru:
  time                  : 2026-08-16T12:15
  interval              : 900
  temperature_2m        : 29.1
  relative_humidity_2m  : 44
  wind_speed_10m        : 19.0


In [3]:
# --- Step 2b: Collect hourly temperature for the next 7 days ---

params2 = {
    "latitude": 12.9716,
    "longitude": 77.5946,
    "hourly": "temperature_2m,relative_humidity_2m,precipitation",
    "timezone": "Asia/Kolkata",
    "forecast_days": 7,
}

try:
    r = requests.get(URL, params=params2, timeout=20)
    result = r.json()
    wx = pd.DataFrame(result["hourly"])       # JSON -> DataFrame
    wx["time"] = pd.to_datetime(wx["time"])
    print("Collected", len(wx), "hourly records")
except Exception as e:
    wx = pd.DataFrame()
    print("No internet — the next cell will create sample data.")

wx.head()

Collected 168 hourly records


,time,temperature_2m,relative_humidity_2m,precipitation
0,2026-08-16 00:00:00,21.7,83,0.0
1,2026-08-16 01:00:00,21.3,86,0.0
2,2026-08-16 02:00:00,20.9,88,0.0
3,2026-08-16 03:00:00,20.7,88,0.0
4,2026-08-16 04:00:00,20.4,89,0.0


In [5]:
WEATHER_URL = "https://api.open-meteo.com/v1/forecast"

params_mysuru = {
    "latitude": 12.2958,
    "longitude": 76.6394,
    "hourly": "temperature_2m"
}

import requests
r = requests.get(WEATHER_URL, params=params_mysuru, timeout=20)
print("Status code:", r.status_code)

weather_mysuru = pd.DataFrame(r.json()["hourly"])
weather_mysuru["time"] = pd.to_datetime(weather_mysuru["time"])
weather_mysuru.to_csv(DATA / "weather_mysuru.csv", index=False)
print("Saved to", DATA / "weather_mysuru.csv")

Status code: 200
Saved to data\weather_mysuru.csv


In [6]:
params_mysuru["hourly"] = "temperature_2m,wind_speed_10m"
r = requests.get(WEATHER_URL, params=params_mysuru, timeout=20)

weather_mysuru = pd.DataFrame(r.json()["hourly"])
weather_mysuru["time"] = pd.to_datetime(weather_mysuru["time"])
weather_mysuru.head()

,time,temperature_2m,wind_speed_10m
0,2026-08-16 00:00:00,20.6,13.9
1,2026-08-16 01:00:00,20.7,13.9
2,2026-08-16 02:00:00,21.9,14.7
3,2026-08-16 03:00:00,23.8,16.5
4,2026-08-16 04:00:00,25.4,18.2


In [7]:
bad_params = params_mysuru.copy()
bad_params["latitude"] = "abcd"
r = requests.get(WEATHER_URL, params=bad_params, timeout=20)
print("Status code:", r.status_code)

Status code: 400


In [10]:
cities = {
    "Bengaluru": (12.9716, 77.5946),
    "Mysuru": (12.2958, 76.6394),
    "Mangaluru": (12.9141, 74.8560),
}
import time
all_weather = []
for city, (lat, lon) in cities.items():
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "temperature_2m,wind_speed_10m"
    }
    r = requests.get(WEATHER_URL, params=params, timeout=20)
    df = pd.DataFrame(r.json()["hourly"])
    df["time"] = pd.to_datetime(df["time"])
    df["city"] = city
    all_weather.append(df)
    time.sleep(1)

weather_all = pd.concat(all_weather, ignore_index=True)
print(weather_all.shape)
weather_all.head()

(504, 4)


,time,temperature_2m,wind_speed_10m,city
0,2026-08-16 00:00:00,20.2,13.2,Bengaluru
1,2026-08-16 01:00:00,20.1,13.5,Bengaluru
2,2026-08-16 02:00:00,21.3,15.5,Bengaluru
3,2026-08-16 03:00:00,23.1,17.8,Bengaluru
4,2026-08-16 04:00:00,25.6,18.4,Bengaluru
